# SZ cross-date diagnostics
Read-only audit of saved diagnostic results. Raw source extraction and independent ledger scripts live in analysis/audit_sz_cross_date_inputs.py, audit_sz_close_price_range.py, audit_sz_sequence_regression.py. Filtered phase-remapped fixtures are causal probes, not full-market production acceptance. No production SZ code was changed.


In [ ]:
import json
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'reports').is_dir() else Path.cwd().parent
out = root / 'reports/20260907-sz-cross-date-diagnosis'
read = lambda name: json.loads((out / name).read_text())
summary = read('summary.json')
assert sum(r.get('mismatched', 0) for r in summary) == 2974
for r in summary: print(r['date'], r['status'], r.get('mismatched'))
close = read('close-range-audit.json')
assert len(close) == 5
assert all(r['ranged_matches'] and r['statistics_match'] and r['independent_full_view_agrees_with_rust'] for r in close)
assert all(r['order_types'] == [50] and r['lifecycle_errors'] == 0 for r in close)


In [ ]:
a, b = read('etf-1s.json'), read('etf-3s.json')
assert a['mismatched'] == 2868 and b['mismatched'] == 0
bad = {(r['symbol'], r['reference_time_ms']) for r in a['records'] if r['outcome'] == 'mismatched'}
hits = [r for r in b['records'] if (r['symbol'], r['reference_time_ms']) in bad]
assert len(hits) == 2868
assert all(r['outcome'] == 'matched' and 1000 <= r['matched_candidate_time_ms'] - r['reference_time_ms'] <= 1040 for r in hits)
print('All 2868 ETF differences: matching candidate T+1000..1040 ms.')


In [ ]:
a, b = read('stock-original.json'), read('stock-v0-aligned.json')
assert a['mismatched'] == 102 and read('stock-v0-remapped.json')['mismatched'] == 102 and b['mismatched'] == 1
bad = [r for r in a['records'] if r['outcome'] == 'mismatched' and r['anchor'] == 'continuous_trading']
current = {r['reference_time_ms']: r for r in b['records'] if r['anchor'] == 'continuous_trading'}
assert len(bad) == 101
for old in bad:
    ts = old['reference_time_ms']
    r = current[ts + 1000 if ts == 1773975504000 else ts]
    assert r['outcome'] == 'matched' and ts <= r['matched_candidate_time_ms'] < ts + 3000
seq = read('sequence-20260616.json')
assert [len(r['regressions']) for r in seq] == [1, 0]
print('101 T0 frames explained by resumption-batch handling; 5 E0 frames by effective price range; original channel regression independently reproduced.')


## Original CSV provenance audit
audit_sz_csv_sequence_origin.py streams the cold archive prefix and compares 111 original CSV rows against raw Parquet. This proves the local inversion predates conversion, not which upstream component caused it. Archive header metadata agrees with provenance; no full-member CRC verification was performed.


In [ ]:
origin = read('csv-origin-audit.json')
assert origin['checked_rows'] == 111
assert origin['all_original_fields_equal'] and origin['inversion_present_in_csv']
rows = {r['source_row_no']: r for r in origin['rows']}
assert int(rows[66683411]['ApplSeqNum']) == 21524148
assert int(rows[66683412]['ApplSeqNum']) == 21524032
assert all(r['ChannelNo'] == '2015' for r in (rows[66683411], rows[66683412]))
print('Original CSV already contains the inversion; all 111 sampled rows agree with Parquet.')
